# Modul 4: Datenexport für Power BI / Data Export for Power BI

**DE:** Dieses Notebook transformiert die bereinigten Rohdaten (`bike_sales_clean.csv`) in ein
optimiertes **Sternschema (Star Schema)** für Power BI.
Jede Ausgabedatei entspricht einer Tabelle im Power BI Datenmodell.

**EN:** This notebook transforms the cleaned raw data (`bike_sales_clean.csv`) into an
optimized **Star Schema** for Power BI.
Each output file corresponds to one table in the Power BI data model.

---

### Sternschema / Star Schema

```
                    ┌─────────────────┐
                    │  pbi_dim_date   │
                    │  (Kalender /    │
                    │   Calendar)     │
                    └────────┬────────┘
                             │ date
┌──────────────────┐         │         ┌──────────────────┐
│ pbi_dim_product  │         │         │ pbi_dim_geography│
│ (Produkt /       ├─────────┼─────────┤ (Land /          │
│  Product)        │ cat/sub │ country │  Country)        │
└──────────────────┘         │         └──────────────────┘
                    ┌────────┴────────┐
                    │ pbi_fact_sales  │
                    │ (Faktentabelle /│
                    │  Fact Table)    │
                    └────────┬────────┘
                             │ year_month
                    ┌────────┴────────┐
                    │  pbi_budget     │
                    │  (Planwerte /   │
                    │   Budget)       │
                    └─────────────────┘
```

---

**Inhaltsverzeichnis / Table of Contents**
1. Setup & Daten laden / Setup & Load Data
2. Faktentabelle / Fact Table (`pbi_fact_sales`)
3. Datumsdimension / Date Dimension (`pbi_dim_date`)
4. Produktdimension / Product Dimension (`pbi_dim_product`)
5. Geographiedimension / Geography Dimension (`pbi_dim_geography`)
6. Budgettabelle / Budget Table (`pbi_budget`)
7. Validierung & Export / Validation & Export

In [ ]:
# =============================================================
# IMPORTS & SETUP
# =============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Output-Pfad / Output path
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Setup abgeschlossen / Setup complete')
print(f'  Output-Verzeichnis / Output directory: {OUTPUT_DIR.resolve()}')

## 1. Setup & Daten laden / Setup & Load Data

**DE:** Wir laden die in Modul 0 aufbereiteten Daten (`bike_sales_clean.csv`).
Diese Datei enthält bereits:
- Bereinigte Spaltennamen / cleaned column names
- Korrekte Datentypen / correct data types
- Neue Spalten (z.B. `gross_margin_pct`, `land_de`, `revenue_plan`) / new columns

**EN:** We load the data prepared in Module 0 (`bike_sales_clean.csv`).
This file already contains:
- Cleaned column names
- Correct data types
- New columns (e.g. `gross_margin_pct`, `land_de`, `revenue_plan`)

In [ ]:
# ------------------------------------------------------------------
# DE: Bereinigte Rohdaten aus Modul 0 laden
# EN: Load cleaned raw data from Module 0
# ------------------------------------------------------------------

df = pd.read_csv('../data/processed/bike_sales_clean.csv')
df['date'] = pd.to_datetime(df['date'])

print('Rohdaten geladen / Raw data loaded:')
print(f'  Zeilen / Rows:    {len(df):,}')
print(f'  Spalten / Cols:   {len(df.columns)}')
print(f'  Zeitraum / Period: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  Länder / Countries: {sorted(df["country"].unique())}')
print(f'  Kategorien / Categories: {df["product_category"].unique().tolist()}')
print()
print('Verfügbare Spalten / Available columns:')
print(df.dtypes.to_string())

## 2. Faktentabelle / Fact Table — `pbi_fact_sales`

**DE:** Die Faktentabelle enthält alle Transaktionen mit den relevanten Kennzahlen.
Sie ist die zentrale Tabelle im Sternschema und wird mit allen Dimensionstabellen verknüpft.

**EN:** The fact table contains all transactions with the relevant KPIs.
It is the central table in the star schema and is joined with all dimension tables.

**Enthaltene Spalten / Included columns:**
- **Schlüssel / Keys:** `date`, `country`, `product_category`, `sub_category`
- **Kennzahlen / Metrics:** `revenue`, `cost`, `profit`, `order_quantity`, `unit_price`, `unit_cost`
- **Plan / Budget:** `revenue_plan`, `cost_plan`, `profit_plan`
- **Abgeleitete Werte / Derived:** `revenue_var_abs`, `revenue_var_pct`, `gross_margin_pct`
- **Zeitfelder / Time fields:** `year_month`, `fiscal_year`, `quarter`, `month_num`

In [ ]:
# ------------------------------------------------------------------
# DE: Faktentabelle aufbauen
# EN: Build fact table
# ------------------------------------------------------------------

# Zeitfelder ergänzen / Add time fields
df['year_month'] = df['date'].dt.to_period('M').astype(str)
df['month_num']  = df['date'].dt.month
df['month_name_de'] = df['month_num'].map({
    1:'Januar', 2:'Februar', 3:'März',    4:'April',
    5:'Mai',    6:'Juni',    7:'Juli',    8:'August',
    9:'September', 10:'Oktober', 11:'November', 12:'Dezember'
})
df['month_name_en'] = df['month_num'].map({
    1:'January',  2:'February', 3:'March',     4:'April',
    5:'May',      6:'June',     7:'July',       8:'August',
    9:'September',10:'October', 11:'November', 12:'December'
})

# Plan-Abweichungen berechnen / Calculate plan variances
df['revenue_var_abs'] = (df['revenue'] - df['revenue_plan']).round(2)
df['revenue_var_pct'] = (
    df['revenue_var_abs'] / df['revenue_plan']
).round(4)

# Faktentabelle zusammenstellen / Assemble fact table
fact_cols = [
    # Schlüssel / Keys
    'date', 'year_month', 'fiscal_year', 'quarter', 'month_num',
    'month_name_de', 'month_name_en',
    'country', 'land_de',
    'product_category', 'sub_category',
    # Mengen / Quantities
    'order_quantity', 'unit_cost', 'unit_price',
    # Finanzkennzahlen Ist / Financial KPIs Actual
    'cost', 'revenue', 'profit', 'gross_margin_pct',
    # Plan / Budget
    'revenue_plan', 'cost_plan', 'profit_plan',
    # Abweichungen / Variances
    'revenue_var_abs', 'revenue_var_pct',
]

fact = df[fact_cols].copy()

print('Faktentabelle / Fact Table — pbi_fact_sales:')
print(f'  Zeilen / Rows:   {len(fact):,}')
print(f'  Spalten / Cols:  {len(fact.columns)}')
print()
print('Spaltenübersicht / Column overview:')
for col in fact.columns:
    print(f'  {col:<22} {str(fact[col].dtype):<12} Beispiel: {fact[col].iloc[0]}')

## 3. Datumsdimension / Date Dimension — `pbi_dim_date`

**DE:** Die Datumsdimension (auch Kalendertabelle genannt) wird in Power BI für
Zeitintelligenz-Funktionen wie `SAMEPERIODLASTYEAR()` oder `TOTALYTD()` benötigt.
Sie wird **vollständig neu generiert** — nicht aus den Rohdaten abgeleitet.

**EN:** The date dimension (also called calendar table) is required in Power BI for
time intelligence functions like `SAMEPERIODLASTYEAR()` or `TOTALYTD()`.
It is **generated from scratch** — not derived from raw data.

In [ ]:
# ------------------------------------------------------------------
# DE: Datumsdimension von Grund auf generieren
# EN: Generate date dimension from scratch
# ------------------------------------------------------------------

# Vollständiger Datumsbereich / Full date range
date_range = pd.date_range(
    start='2011-01-01',
    end='2016-12-31',
    freq='D'  # täglich / daily
)

dim_date = pd.DataFrame({'date': date_range})

# Zeitfelder ableiten / Derive time fields
dim_date['year']         = dim_date['date'].dt.year
dim_date['quarter']      = 'Q' + dim_date['date'].dt.quarter.astype(str)
dim_date['quarter_num']  = dim_date['date'].dt.quarter
dim_date['month_num']    = dim_date['date'].dt.month
dim_date['month_name_de']= dim_date['month_num'].map({
    1:'Januar', 2:'Februar', 3:'März',    4:'April',
    5:'Mai',    6:'Juni',    7:'Juli',    8:'August',
    9:'September', 10:'Oktober', 11:'November', 12:'Dezember'
})
dim_date['month_name_en']= dim_date['month_num'].map({
    1:'January', 2:'February', 3:'March',  4:'April',
    5:'May',     6:'June',     7:'July',   8:'August',
    9:'September',10:'October',11:'November',12:'December'
})
dim_date['year_month']   = dim_date['date'].dt.to_period('M').astype(str)
dim_date['week_num']     = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['day_of_week']  = dim_date['date'].dt.day_name()  # Monday, Tuesday...
dim_date['is_weekend']   = dim_date['date'].dt.dayofweek >= 5
dim_date['day_num']      = dim_date['date'].dt.day
# Fiskalquartal-Label / Fiscal quarter label
dim_date['year_quarter'] = (
    dim_date['year'].astype(str) + ' ' + dim_date['quarter']
)

print('Datumsdimension / Date Dimension — pbi_dim_date:')
print(f'  Zeilen / Rows:   {len(dim_date):,} (täglich / daily)')
print(f'  Spalten / Cols:  {len(dim_date.columns)}')
print(f'  Von / From:      {dim_date["date"].min().date()}')
print(f'  Bis / To:        {dim_date["date"].max().date()}')
print()
print('Vorschau / Preview:')
print(dim_date.head(3).to_string())

# Warum eine eigene Datumstabelle? / Why a separate date table?
print()
print('💡 Warum Datumsdimension? / Why date dimension?')
print('   DE: Power BI Zeitintelligenz (YoY, MTD, YTD) benötigt eine lückenlose Kalendertabelle.')
print('   EN: Power BI time intelligence (YoY, MTD, YTD) requires a contiguous calendar table.')
print('   DAX-Funktionen: SAMEPERIODLASTYEAR(), TOTALYTD(), DATEADD()')

## 4. Produktdimension / Product Dimension — `pbi_dim_product`

**DE:** Die Produktdimension enthält alle eindeutigen Produkt-Kombinationen
(Kategorie + Unterkategorie). Sie wird direkt aus den Rohdaten abgeleitet.

**EN:** The product dimension contains all unique product combinations
(category + subcategory). It is derived directly from the raw data.

In [ ]:
# ------------------------------------------------------------------
# DE: Produktdimension aus Rohdaten ableiten
# EN: Derive product dimension from raw data
# ------------------------------------------------------------------

dim_product = (
    df[['product_category', 'sub_category']]
    .drop_duplicates()
    .sort_values(['product_category', 'sub_category'])
    .reset_index(drop=True)
)

# Eindeutiger Schlüssel / Unique key
dim_product['product_key'] = range(1, len(dim_product) + 1)

# Kategorie auf Deutsch / Category in German
dim_product['category_de'] = dim_product['product_category'].map({
    'Accessories': 'Zubehör',
    'Bikes':       'Fahrräder',
    'Clothing':    'Bekleidung'
})

# Margen-Ziel pro Kategorie / Margin target per category
dim_product['margin_target'] = dim_product['product_category'].map({
    'Accessories': 0.55,
    'Bikes':       0.35,
    'Clothing':    0.40
})

# Spalten neu ordnen / Reorder columns
dim_product = dim_product[[
    'product_key', 'product_category', 'category_de',
    'sub_category', 'margin_target'
]]

print('Produktdimension / Product Dimension — pbi_dim_product:')
print(f'  Zeilen / Rows:   {len(dim_product)}')
print(f'  Spalten / Cols:  {len(dim_product.columns)}')
print()
print(dim_product.to_string(index=False))

## 5. Geographiedimension / Geography Dimension — `pbi_dim_geography`

**DE:** Die Geographiedimension enthält alle Länder mit deutschen Bezeichnungen
und einer zusätzlichen Regionsgruppierung für übergeordnete Analysen.

**EN:** The geography dimension contains all countries with German names
and an additional region grouping for higher-level analysis.

In [ ]:
# ------------------------------------------------------------------
# DE: Geographiedimension aufbauen
# EN: Build geography dimension
# ------------------------------------------------------------------

dim_geo = (
    df[['country', 'land_de']]
    .drop_duplicates()
    .sort_values('country')
    .reset_index(drop=True)
)

# Eindeutiger Schlüssel / Unique key
dim_geo['geo_key'] = range(1, len(dim_geo) + 1)

# Region / Region grouping
dim_geo['region_en'] = dim_geo['country'].map({
    'Australia':      'Asia-Pacific',
    'Canada':         'North America',
    'France':         'Europe',
    'Germany':        'Europe',
    'United Kingdom': 'Europe',
    'United States':  'North America'
})
dim_geo['region_de'] = dim_geo['region_en'].map({
    'Asia-Pacific':  'Asien-Pazifik',
    'North America': 'Nordamerika',
    'Europe':        'Europa'
})

# ISO-Code
dim_geo['iso_code'] = dim_geo['country'].map({
    'Australia':      'AU',
    'Canada':         'CA',
    'France':         'FR',
    'Germany':        'DE',
    'United Kingdom': 'GB',
    'United States':  'US'
})

# Spalten ordnen / Order columns
dim_geo = dim_geo[[
    'geo_key', 'iso_code', 'country', 'land_de',
    'region_en', 'region_de'
]]

print('Geographiedimension / Geography Dimension — pbi_dim_geography:')
print(f'  Zeilen / Rows:   {len(dim_geo)}')
print(f'  Spalten / Cols:  {len(dim_geo.columns)}')
print()
print(dim_geo.to_string(index=False))

## 6. Budgettabelle / Budget Table — `pbi_budget`

**DE:** Die Budgettabelle enthält aggregierte Planwerte pro Monat, Kategorie und Land.
Sie wird in Power BI mit der Faktentabelle über `year_month`, `product_category`
und `country` verknüpft.

**EN:** The budget table contains aggregated plan values per month, category and country.
In Power BI it joins with the fact table via `year_month`, `product_category` and `country`.

**Woher kommen die Planwerte? / Where do budget values come from?**  
DE: In Modul 0 wurden die Planwerte simuliert: `revenue_plan = revenue × (1 ± 12%)`.  
Dies ist die Standardmethode in der Praxis wenn historische Budgetdaten fehlen.  
EN: In Module 0, plan values were simulated: `revenue_plan = revenue × (1 ± 12%)`.  
This is standard practice when historical budget data is unavailable.

In [ ]:
# ------------------------------------------------------------------
# DE: Budgettabelle durch Aggregation aufbauen
# EN: Build budget table via aggregation
# ------------------------------------------------------------------

budget = (
    df
    .groupby(['fiscal_year', 'year_month', 'product_category', 'country'])
    .agg(
        revenue_plan = ('revenue_plan', 'sum'),
        cost_plan    = ('cost_plan',    'sum'),
        profit_plan  = ('profit_plan',  'sum'),
    )
    .reset_index()
)

# Abgeleitete Plan-Kennzahlen / Derived budget KPIs
budget['margin_plan_pct'] = (
    budget['profit_plan'] / budget['revenue_plan']
).round(4)

# Sortierung / Sort
budget = budget.sort_values(
    ['fiscal_year', 'year_month', 'country', 'product_category']
).reset_index(drop=True)

print('Budgettabelle / Budget Table — pbi_budget:')
print(f'  Zeilen / Rows:   {len(budget):,}')
print(f'  Spalten / Cols:  {len(budget.columns)}')
print()
print('Vorschau 2015 / Preview 2015 (erste 6 Zeilen / first 6 rows):')
print(
    budget[budget['fiscal_year'] == 2015]
    .head(6)
    .to_string(index=False)
)
print()
print('💡 Warum aggregiert? / Why aggregated?')
print('   DE: Power BI verknüpft Budget und Ist auf Monats-/Kategorie-Ebene — nicht auf Transaktionsebene.')
print('   EN: Power BI joins Budget and Actual at month/category level — not at transaction level.')

## 7. Validierung & Export / Validation & Export

**DE:** Vor dem Export prüfen wir alle Tabellen auf Konsistenz:
- Schlüsselfelder vollständig / Key fields complete
- Keine doppelten Primärschlüssel / No duplicate primary keys
- Summen stimmen mit Rohdaten überein / Totals match raw data

**EN:** Before export we validate all tables for consistency:
- Key fields complete
- No duplicate primary keys
- Totals match raw data

In [ ]:
# ------------------------------------------------------------------
# DE: Validierung aller Tabellen
# EN: Validate all tables
# ------------------------------------------------------------------

print('=' * 65)
print('VALIDIERUNG / VALIDATION')
print('=' * 65)

# ── Fact Table ──
print('\n① Faktentabelle / Fact Table (pbi_fact_sales):')
print(f'  Zeilen / Rows:           {len(fact):,}')
print(f'  Fehlende Werte / Nulls:  {fact.isnull().sum().sum()}')
print(f'  Revenue Gesamt / Total:  {fact["revenue"].sum():,.0f} USD')
print(f'  Profit Gesamt / Total:   {fact["profit"].sum():,.0f} USD')
print(f'  Marge / Margin:          {fact["profit"].sum()/fact["revenue"].sum()*100:.1f}%')

# ── Date Dimension ──
print('\n② Datumsdimension / Date Dimension (pbi_dim_date):')
print(f'  Zeilen / Rows:           {len(dim_date):,}')
print(f'  Eindeutig / Unique dates: {dim_date["date"].nunique():,}')
print(f'  Doppelt / Duplicates:    {dim_date["date"].duplicated().sum()}')
print(f'  Lücken / Gaps:           {((dim_date["date"].max() - dim_date["date"].min()).days + 1) - len(dim_date)}')

# ── Product Dimension ──
print('\n③ Produktdimension / Product Dimension (pbi_dim_product):')
print(f'  Zeilen / Rows:           {len(dim_product)}')
print(f'  Doppelte Keys / Dup keys:{dim_product["product_key"].duplicated().sum()}')
fact_cats = set(fact['product_category'].unique())
dim_cats  = set(dim_product['product_category'].unique())
print(f'  Alle Kategorien gemappt: {fact_cats == dim_cats} (Fact ↔ Dim)')

# ── Geography Dimension ──
print('\n④ Geographiedimension / Geography Dimension (pbi_dim_geography):')
print(f'  Zeilen / Rows:           {len(dim_geo)}')
print(f'  Doppelte Keys / Dup keys:{dim_geo["geo_key"].duplicated().sum()}')
fact_ctr = set(fact['country'].unique())
dim_ctr  = set(dim_geo['country'].unique())
print(f'  Alle Länder gemappt:     {fact_ctr == dim_ctr} (Fact ↔ Dim)')

# ── Budget Table ──
print('\n⑤ Budgettabelle / Budget Table (pbi_budget):')
print(f'  Zeilen / Rows:           {len(budget):,}')
print(f'  Plan Revenue Total:      {budget["revenue_plan"].sum():,.0f} USD')
print(f'  Plan Profit Total:       {budget["profit_plan"].sum():,.0f} USD')

# Kontrolle: Fact vs. Raw ──
raw_rev = df['revenue'].sum()
fct_rev = fact['revenue'].sum()
print(f'\n⑥ Kontrolle Fact vs. Raw / Cross-check Fact vs. Raw:')
print(f'  Rohdaten Revenue / Raw revenue: {raw_rev:,.0f} USD')
print(f'  Fact Revenue:                   {fct_rev:,.0f} USD')
print(f'  Differenz / Difference:         {abs(raw_rev - fct_rev):,.0f} USD')
print(f'  Status: {"✅ OK" if abs(raw_rev - fct_rev) < 1 else "❌ FEHLER / ERROR"}')

print('\n' + '=' * 65)
print('Validierung abgeschlossen / Validation complete')
print('=' * 65)

In [ ]:
# ------------------------------------------------------------------
# DE: Alle Tabellen als CSV exportieren
# EN: Export all tables as CSV
# ------------------------------------------------------------------

export_files = {
    'pbi_fact_sales.csv':       fact,
    'pbi_dim_date.csv':         dim_date,
    'pbi_dim_product.csv':      dim_product,
    'pbi_dim_geography.csv':    dim_geo,
    'pbi_budget.csv':           budget,
}

print('Export gestartet / Export started...')
print()

for filename, table in export_files.items():
    filepath = OUTPUT_DIR / filename
    table.to_csv(filepath, index=False, encoding='utf-8-sig')
    size_kb = filepath.stat().st_size / 1024
    print(f'  ✓ {filename:<35} {len(table):>8,} Zeilen / rows   {size_kb:>8.1f} KB')

print()
print('=' * 65)
print('Export abgeschlossen / Export complete')
print('=' * 65)
print()
print('Power BI Import-Reihenfolge / Power BI Import Order:')
print('  1. pbi_dim_date.csv          → Datumsdimension / Date dim')
print('  2. pbi_dim_product.csv       → Produktdimension / Product dim')
print('  3. pbi_dim_geography.csv     → Länderdimension / Geography dim')
print('  4. pbi_budget.csv            → Planwerte / Budget values')
print('  5. pbi_fact_sales.csv        → Faktentabelle (zuletzt!) / Fact table (last!)')
print()
print('Nächster Schritt / Next step: 03_powerbi_dashboard_guide.md öffnen und folgen!')